# Individual Test Run Recon

Analyse the full `transactions` table for a single test execution.

---
## 1. Init

Modify `RESULTS_BASE_DIR` to traverse through subfolders

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

from faultlab.analysis import (
    load_csv,
    enrich_latency_metrics,
    describe_latency_percentiles,
    execution_summary,
    status_summary,
    FAILURE_STATUSES,
    LATENCY_COLS,
    TERMINAL_STATUSES,
    STATUS_COLORS,
)

# == Plotting =====================================================================
sns.set_theme(style="whitegrid")

# == Paths =====================================================================
PROJECT_ROOT_DIR = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT_DIR.joinpath("src")))

RESULTS_BASE_DIR = "results/BASELINE"
# RESULTS_BASE_DIR = "results/GRPC_CONCURRENCY_LIMIT/1"

RESULTS_DIR = PROJECT_ROOT_DIR.joinpath("tests").joinpath(RESULTS_BASE_DIR)

print(f"Project root : {PROJECT_ROOT_DIR}")
print(f"Results dir  : {RESULTS_DIR}")

---
### 2. Load File

Set `FILE_NAME` variable

In [ ]:
FILE_NAME = "test_baseline_1000_txs_10_accounts_20260531_204144.csv"

CSV_FILE = RESULTS_DIR.joinpath(f"{FILE_NAME}{"" if FILE_NAME.endswith(".csv") else ".csv"}")
assert CSV_FILE and CSV_FILE.exists(), f"File not found: {CSV_FILE}"

In [ ]:
df = load_csv(CSV_FILE)

print(f"Loaded {len(df)} rows from {CSV_FILE.name}")
display(df.head(5))

---
### 3. Execution Overview

Shows main accumulated test metrics

In [ ]:
df = enrich_latency_metrics(df)
df_signed = df[df["signed_at"].notna()].copy()


summary_df = execution_summary(df)
display(summary_df)

---
### 4. Stage Breakdown

Durations are computed for each pipeline stage:
- **Full signing latency (Including Queue Time)**   — `created_at` -> `signed_at`
- **Pure signing latency (Excluding Queue Time)**   — `signing_started_at` -> `signed_at`
- **Submission latency**                            — `signed_at` -> `submitted_at`
- **Confirmation latency**                          — `submitted_at` -> `confirmed_at`
- **End-to-end latency**                            — `created_at` -> `confirmed_at`

In [ ]:
lat_cols = [c for c in LATENCY_COLS if c in df.columns]

display(describe_latency_percentiles(df, TERMINAL_STATUSES).round(4))

---
### 5. Status Distribution

In [ ]:
summary = status_summary(df)
display(summary)

colors = [STATUS_COLORS.get(s, "#cccccc") for s in summary.index]
_, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
bars = ax1.bar(summary.index, summary["count"], color=colors)
ax1.bar_label(bars, fmt="%d")
ax1.set_title("Transaction Count by Status", fontweight="bold")
ax1.set_xlabel("Status")
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=30)

# Pie chart
nonzero = summary[summary["count"] > 0]
pie_colors = [STATUS_COLORS.get(s, "#cccccc") for s in nonzero.index]
ax2.pie(
    nonzero["count"],
    labels=nonzero.index,
    autopct="%1.1f%%",
    colors=pie_colors,
    startangle=140,
)
ax2.set_title("Status Distribution (%)", fontweight="bold")

plt.show()

---
### 6. Recovery Analysis 

In [ ]:
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=df_signed,
    x="signing_retries",
    y="pure_signing_latency_s",
    estimator=np.median,
    errorbar=("pi", 95),
    marker="o",
    linewidth=2,
)

zero_retries = df_signed[df_signed["signing_retries"] == 0]
if not zero_retries.empty:
    baseline_median = zero_retries["pure_signing_latency_s"].median()
    plt.axhline(y=baseline_median, color="green", linestyle="--", label=f"Healthy Baseline ({baseline_median:.2f}s)")
    plt.legend()

max_retries = int(df_signed["signing_retries"].max()) if not df_signed.empty else 0
plt.xticks(range(0, max_retries + 1))

plt.title("Mean Time To Recovery", fontweight="bold")
plt.xlabel("Signing Retries")
plt.ylabel("Mean Signing Duration (s)")
plt.show()

---
### 7. State Consistency Analysis

In [ ]:
# 1. Normalize timestamps
df["signed_sec"] = pd.to_datetime(df["signed_at"], utc=True).dt.floor("s")
df["submitted_sec"] = pd.to_datetime(df["submitted_at"], utc=True).dt.floor("s")
df["confirmed_sec"] = pd.to_datetime(df["confirmed_at"], utc=True).dt.floor("s")

# 2. Count transactions per second
signed_series = df.groupby("signed_sec")["id"].count()
submitted_series = df.groupby("submitted_sec")["id"].count()
confirmed_series = df.groupby("confirmed_sec")["id"].count()

# 3. Create master timeline
start_time = min(df["signed_sec"].min(), df["submitted_sec"].min(), df["confirmed_sec"].min())
end_time = max(df["signed_sec"].max(), df["submitted_sec"].max(), df["confirmed_sec"].max())
full_timeline = pd.date_range(start=start_time, end=end_time, freq="s")

# 4. Calculate cumulative sums
cumulative_df = pd.DataFrame(index=full_timeline)
cumulative_df["Total Signed"] = signed_series.reindex(full_timeline, fill_value=0).cumsum()
cumulative_df["Total Submitted"] = submitted_series.reindex(full_timeline, fill_value=0).cumsum()
cumulative_df["Total Confirmed"] = confirmed_series.reindex(full_timeline, fill_value=0).cumsum()

# 5. Convert to relative time
cumulative_df["seconds_from_start"] = (cumulative_df.index - start_time).total_seconds()

# 6. Plotting
plt.figure(figsize=(12, 6))

# The three boundary lines
plt.plot(
    cumulative_df["seconds_from_start"],
    cumulative_df["Total Signed"],
    color="#5614c7",
    label="Cumulative Signed",
)
plt.plot(
    cumulative_df["seconds_from_start"],
    cumulative_df["Total Submitted"],
    color="#ffb300",
    label="Cumulative Submitted",
)
plt.plot(
    cumulative_df["seconds_from_start"],
    cumulative_df["Total Confirmed"],
    color="#388e3c",
    label="Cumulative Confirmed",
)

# Fill 1: Internal Java Queue
plt.fill_between(
    cumulative_df["seconds_from_start"],
    cumulative_df["Total Submitted"],
    cumulative_df["Total Signed"],
    color="#5614c7",
    alpha=0.15,
    label="Internal Queue (Waiting for Submission)",
)

# Fill 2: Hardhat Network Mempool
plt.fill_between(
    cumulative_df["seconds_from_start"],
    cumulative_df["Total Confirmed"],
    cumulative_df["Total Submitted"],
    color="#ffb300",
    alpha=0.25,
    label="Network Mempool (Waiting to Mine)",
)

plt.title("System State Consistency", fontweight="bold")
plt.xlabel("Seconds from Start")
plt.ylabel("Total Transactions")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()